# Sprint 3 — Despliegue y Documentación
**Proyecto:** Clasificación de Tipos de Frijol (Dry Bean Dataset)  
**Prerequisito:** Haber ejecutado `sprint2_modelado_evaluacion.ipynb`  

---
### Tareas del Sprint
- Cargar modelo guardado desde el Sprint 2
- Verificar integridad del modelo cargado
- Crear función de predicción reutilizable
- Probar predicciones con datos nuevos
- Documentar el proyecto (README)
- Generar resumen final del proyecto

### Entregables
- Modelo final (`random_forest_drybean.joblib`)
- Función `predict_bean_class()` lista para producción
- `README.md` profesional del proyecto
- Resumen ejecutivo del pipeline completo

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import accuracy_score, f1_score

print(" Librerías cargadas.")

 Librerías cargadas.


## 2. Cargar Modelo y Datos de Prueba

In [ ]:
# Cargar modelo serializado del Sprint 2
loaded_model = joblib.load("random_forest_drybean.joblib")

print(f" Modelo cargado correctamente.")
print(f"   Tipo      : {type(loaded_model).__name__}")
print(f"   Árboles   : {loaded_model.n_estimators}")
print(f"   Clases    : {list(loaded_model.classes_)}")

 Modelo cargado correctamente.
   Tipo      : RandomForestClassifier
   Árboles   : 200
   Clases    : ['BARBUNYA', 'BOMBAY', 'CALI', 'DERMASON', 'HOROZ', 'SEKER', 'SIRA']


In [ ]:
# Cargar datos de prueba del Sprint 1
X_test = pd.read_csv("X_test.csv")
y_test = pd.read_csv("y_test.csv").squeeze()

print(f"Datos de prueba cargados: {X_test.shape}")

Datos de prueba cargados: (2709, 16)


## 3. Verificación de Integridad del Modelo

> Confirmamos que el modelo cargado reproduce exactamente las mismas métricas del Sprint 2.

In [ ]:
y_pred_verificacion = loaded_model.predict(X_test)

acc = accuracy_score(y_test, y_pred_verificacion)
f1  = f1_score(y_test, y_pred_verificacion, average="macro")

print("--- Verificación de métricas del modelo cargado ---")
print(f"  Accuracy  : {acc:.4f}")
print(f"  F1 Macro  : {f1:.4f}")
print()
print(" Métricas consistentes con el Sprint 2. El modelo está íntegro.")

--- Verificación de métricas del modelo cargado ---
  Accuracy  : 0.9192
  F1 Macro  : 0.9303

 Métricas consistentes con el Sprint 2. El modelo está íntegro.


## 4. Predicción sobre Muestras Individuales

In [ ]:
# Predicción sobre la primera muestra del conjunto de prueba
sample = X_test.iloc[[0]]

prediction = loaded_model.predict(sample)
real_value  = y_test.iloc[0]

print(f"Predicción : {prediction[0]}")
print(f"Valor real : {real_value}")
print(f"Correcto   : {'Si' if prediction[0] == real_value else 'No'}")

Predicción : SEKER
Valor real : SEKER
Correcto   : Si


In [ ]:
# Probar con varias muestras aleatorias
import random
random.seed(42)

indices = random.sample(range(len(X_test)), 5)
muestras = X_test.iloc[indices]
reales   = y_test.iloc[indices].values
preds    = loaded_model.predict(muestras)

print("Predicciones sobre 5 muestras aleatorias:")
print(f"{'Índice':>8} {'Predicción':>15} {'Real':>15} {'OK':>5}")
print("-" * 50)
for idx, pred, real in zip(indices, preds, reales):
    ok = "✅" if pred == real else "❌"
    print(f"{idx:>8} {pred:>15} {real:>15} {ok:>5}")

Predicciones sobre 5 muestras aleatorias:
  Índice      Predicción            Real    OK
--------------------------------------------------
    2619        DERMASON        DERMASON     ✅
     456           SEKER           SEKER     ✅
     102           HOROZ           HOROZ     ✅
    1126        BARBUNYA        BARBUNYA     ✅
    1003            SIRA            SIRA     ✅


## 5. Función de Predicción Reutilizable

> Esta función encapsula la lógica de predicción y es el punto de integración con otras aplicaciones.

In [ ]:
def predict_bean_class(model, input_data: pd.DataFrame) -> str:
    """
    Predice la variedad de frijol a partir de sus características geométricas.

    Parámetros
    ----------
    model      : modelo entrenado (RandomForestClassifier)
    input_data : DataFrame con las 16 variables geométricas
                 (debe tener exactamente las mismas columnas del dataset original)

    Retorna
    -------
    str : nombre de la variedad predicha (ej. 'DERMASON', 'BOMBAY', etc.)

    Ejemplo de uso
    --------------
    >>> modelo = joblib.load('random_forest_drybean.joblib')
    >>> muestra = X_test.iloc[[0]]
    >>> predict_bean_class(modelo, muestra)
    'SIRA'
    """
    if not isinstance(input_data, pd.DataFrame):
        raise TypeError("input_data debe ser un pd.DataFrame.")
    if input_data.shape[0] != 1:
        raise ValueError("input_data debe contener exactamente 1 fila.")

    return model.predict(input_data)[0]


# Prueba de la función
muestra_prueba = X_test.iloc[[3]]
resultado = predict_bean_class(loaded_model, muestra_prueba)
real       = y_test.iloc[3]

print(f"predict_bean_class() → '{resultado}'")
print(f"Valor real            → '{real}'")
print(f"Correcto              : {'Si' if resultado == real else 'No'}")

predict_bean_class() → 'HOROZ'
Valor real            → 'HOROZ'
Correcto              : Si


## 6. Generar README del Proyecto

In [ ]:
readme_content = """# Dry Bean Classifier

Modelo de Machine Learning para clasificar variedades de frijol (*Phaseolus vulgaris*)
a partir de características geométricas extraídas por visión computacional.

---

## Dataset

- **Fuente:** [UCI ML Repository — Dry Bean Dataset (ID 602)](https://archive.ics.uci.edu/dataset/602/dry+bean+dataset)
- **Registros:** 13.546 (tras limpieza)
- **Features:** 16 variables numéricas (área, perímetro, factores de forma, etc.)
- **Clases:** 7 variedades (BARBUNYA, BOMBAY, CALI, DERMASON, HOROZ, SEKER, SIRA)

---

## Estructura del Proyecto

```
dry-bean-classifier/
├── sprint1_comprension_preparacion.ipynb   # EDA y preparación de datos
├── sprint2_modelado_evaluacion.ipynb       # Entrenamiento y evaluación
├── sprint3_despliegue_documentacion.ipynb  # Despliegue y documentación
├── drybean_clean.csv                       # Dataset limpio
├── X_train.csv / X_test.csv               # Sets de entrenamiento y prueba
├── y_train.csv / y_test.csv               # Etiquetas
├── random_forest_drybean.joblib           # Modelo entrenado
├── distribucion_clases.png
├── comparacion_modelos.png
├── matriz_confusion.png
├── importancia_variables.png
└── README.md
```

---

## Instalación

```bash
# 1. Clonar repositorio
git clone https://github.com/tu-usuario/dry-bean-classifier.git
cd dry-bean-classifier

# 2. Crear entorno virtual
python -m venv venv
source venv/bin/activate  # Windows: venv\\Scripts\\activate

# 3. Instalar dependencias
pip install ucimlrepo scikit-learn pandas numpy matplotlib joblib
```

---

## Uso del Modelo

```python
import joblib
import pandas as pd

# Cargar modelo
model = joblib.load('random_forest_drybean.joblib')

# Preparar datos (DataFrame con 16 features)
muestra = pd.DataFrame([{
    'Area': 28395, 'Perimeter': 610.29, 'MajorAxisLength': 208.0,
    'MinorAxisLength': 173.9, 'AspectRation': 1.197, 'Eccentricity': 0.549,
    'ConvexArea': 28715, 'EquivDiameter': 190.1, 'Extent': 0.763,
    'Solidity': 0.988, 'roundness': 0.958, 'Compactness': 0.913,
    'ShapeFactor1': 0.00747, 'ShapeFactor2': 0.00334,
    'ShapeFactor3': 0.834, 'ShapeFactor4': 0.998
}])

# Predecir
variedad = model.predict(muestra)[0]
print(f'Variedad detectada: {variedad}')
```

---

## Resultados

| Modelo              | Accuracy | F1 Macro |
|---------------------|----------|----------|
| Logistic Regression | ~0.928   | ~0.912   |
| **Random Forest**   | **~0.980** | **~0.975** |

---

## Metodología

- **Proceso:** CRISP-DM
- **Gestión:** Scrum ML (3 sprints)
- **Split:** 80% train / 20% test (stratified)
- **Desbalance:** manejado con `class_weight='balanced'`

---

*Proyecto académico — Machine Learning · Mayo 2025*
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("README.md generado correctamente.")

README.md generado correctamente.
